In [ ]:
# Importing necessary libraries for data manipulation and visualization
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Importing preprocessing and model classes from sklearn
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, MinMaxScaler 
from sklearn.linear_model import LogisticRegression, Lasso, Ridge

# Importing metrics for model evaluation
from sklearn.metrics import mean_squared_error, confusion_matrix, accuracy_score 
# Importing train_test_split for splitting the dataset into training and testing sets
from sklearn.model_selection import train_test_split
# Importing RandomForestClassifier for classification tasks
from sklearn.ensemble import RandomForestClassifier

# Suppressing warnings for cleaner output
import warnings 
warnings.filterwarnings("ignore")

# Defining a class to hold color codes for terminal output
class color:
   PURPLE = '\033[95m'      # Purple color code
   CYAN = '\033[96m'        # Cyan color code
   DARKCYAN = '\033[36m'    # Dark Cyan color code
   BLUE = '\033[94m'        # Blue color code
   GREEN = '\033[92m'       # Green color code
   YELLOW = '\033[93m'      # Yellow color code
   RED = '\033[91m'         # Red color code
   BOLD = '\033[1m'         # Bold text code
   UNDERLINE = '\033[4m'    # Underlined text code
   END = '\033[0m'          # Reset color code

In [ ]:
# 2) الاتصال بقاعدة SQL Server وتحميل جدول train
import pyodbc
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

connection_string = (
[Connection details redacted for public portfolio]
[Connection details redacted for public portfolio]
[Connection details redacted for public portfolio]
[Connection details redacted for public portfolio]
    "TrustServerCertificate=yes;"
)
connection_url = URL.create(
    "mssql+pyodbc",
[Connection details redacted for public portfolio]
)
[Connection details redacted for public portfolio]

df = pd.read_sql("SELECT * FROM dbo.[train]", engine)
print("Loaded shape:", df.shape)
display(df.head())
# توحيد أسماء الأعمدة القادمة من SQL
df.columns = df.columns.astype(str).str.strip().str.replace(" ", "_", regex=False)
print("Columns:", df.columns.tolist())


In [ ]:
# 3) تجهيز أنواع البيانات
df["Order_Date"] = pd.to_datetime(df["Order_Date"], dayfirst=True, errors="coerce")
df["Ship_Date"] = pd.to_datetime(df["Ship_Date"], dayfirst=True, errors="coerce")
df["Sales"] = pd.to_numeric(df["Sales"], errors="coerce")

# استبعاد الصفوف التي لا تحتوي على تاريخ طلب أو قيمة مبيعات قابلة للتحليل
df = df.dropna(subset=["Order_Date", "Sales"]).copy()
df["Order_Month"] = df["Order_Date"].dt.to_period("M").dt.to_timestamp()
df["Order_Year"] = df["Order_Date"].dt.year

print("Rows after basic preparation:", len(df))
print("Date range:", df["Order_Date"].min().date(), "to", df["Order_Date"].max().date())
print("Missing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("Missing"))

In [ ]:
display(df[df["Postal_Code"].isna()])

In [ ]:
# 4) مؤشرات الأداء الأساسية (KPIs)
total_sales = df["Sales"].sum()
order_count = df["Order_ID"].nunique()
customer_count = df["Customer_ID"].nunique()
average_order_value = df.groupby("Order_ID")["Sales"].sum().mean()

print(f"Total sales: ${total_sales:,.2f}")
print(f"Unique orders: {order_count:,}")
print(f"Unique customers: {customer_count:,}")
print(f"Average sales per order: ${average_order_value:,.2f}")

In [ ]:
# 5) اتجاه المبيعات شهريًا
monthly_sales = (
    df.groupby("Order_Month", as_index=False)["Sales"].sum()
      .sort_values("Order_Month")
)
plt.figure(figsize=(14, 5))
sns.lineplot(data=monthly_sales, x="Order_Month", y="Sales", marker="o")
plt.title("Monthly Sales Trend")
plt.xlabel("Order_Month")
plt.ylabel("Sales ($)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

display(monthly_sales.tail(12))

In [ ]:
# 6) مقارنة السنوات — هل المبيعات تتحسن أم تتراجع؟
yearly_sales = df.groupby("Order_Year", as_index=False)["Sales"].sum()
yearly_sales["YoY_Change_%"] = yearly_sales["Sales"].pct_change() * 100
display(yearly_sales)

plt.figure(figsize=(9, 5))
sns.barplot(data=yearly_sales, x="Order_Year", y="Sales", errorbar=None)
plt.title("Total Sales by Year")
plt.xlabel("Year")
plt.ylabel("Sales ($)")
plt.tight_layout()
plt.show()

In [ ]:
# 7) تحديد المناطق الأضعف بالمبيعات
region_sales = (
    df.groupby("Region", as_index=False)
      .agg(Sales=("Sales", "sum"), Orders=("Order_ID", "nunique"))
      .sort_values("Sales")
)
display(region_sales)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.barplot(data=region_sales, x="Sales", y="Region", ax=axes[0], errorbar=None)
axes[0].set_title("Sales by Region")
sns.barplot(data=region_sales, x="Orders", y="Region", ax=axes[1], errorbar=None)
axes[1].set_title("Unique Orders by Region")
plt.tight_layout()
plt.show()

In [ ]:
# 8) الفئات والفئات الفرعية الأقل مبيعًا

category_sales = (
    df.groupby("Category", as_index=False)
      .agg(
          Sales=("Sales", "sum"),
          Orders=("Order_ID", "nunique")
      )
      .sort_values("Sales")
)

subcat_sales = (
    df.groupby("Sub_Category", as_index=False)
      .agg(
          Sales=("Sales", "sum"),
          Orders=("Order_ID", "nunique")
      )
      .sort_values("Sales")
)

display(category_sales)
display(subcat_sales.head(10))

In [ ]:
# 9) المنتجات الأقل والأعلى مبيعًا
product_sales = (
    df.groupby(["Product_ID", "Product_Name"], as_index=False)
      .agg(Sales=("Sales", "sum"), Orders=("Order_ID", "nunique"))
      .sort_values("Sales")
)
print("10 lowest-sales products")
display(product_sales.head(10))
print("10 highest-sales products")
display(product_sales.tail(10).sort_values("Sales", ascending=False))

In [ ]:
# 10) التمييز بين انخفاض قيمة المبيعات وانخفاض عدد الطلبات
monthly_orders = (
    df.groupby("Order_Month")
      .agg(Sales=("Sales", "sum"), Orders=("Order_ID", "nunique"))
      .reset_index()
)
monthly_orders["Sales per Order"] = monthly_orders["Sales"] / monthly_orders["Orders"]
display(monthly_orders.tail(12))

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
sns.lineplot(data=monthly_orders, x="Order_Month", y="Sales", marker="o", ax=axes[0])
axes[0].set_title("Monthly Sales")
sns.lineplot(data=monthly_orders, x="Order_Month", y="Orders", marker="o", ax=axes[1])
axes[1].set_title("Monthly Unique Orders")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# 11) تحديد أشهر الانخفاض مقارنة بالشهر السابق
monthly_orders["Previous_Month_Sales"] = monthly_orders["Sales"].shift(1)
monthly_orders["MoM_Change_%"] = (
    (monthly_orders["Sales"] - monthly_orders["Previous_Month_Sales"])
    / monthly_orders["Previous_Month_Sales"] * 100
)
declines = monthly_orders[monthly_orders["MoM_Change_%"] < 0].sort_values("MoM_Change_%")
display(declines[["Order_Month", "Sales", "Previous_Month_Sales", "MoM_Change_%"]].head(10))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# تجهيز البيانات
heatmap_data = df.pivot_table(
    index="Region",
    columns="Category",
    values="Sales",
    aggfunc="sum",
    fill_value=0
)

# رسم الخريطة الحرارية
plt.figure(figsize=(12, 7))

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=",.0f",
    cmap="YlOrRd",
    linewidths=1,
    linecolor="white",
    cbar_kws={"label": "Total Sales"}
)

plt.title(
    "Sales Heatmap by Region and Category",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Product Category")
plt.ylabel("Region")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))

sns.boxplot(
    data=df,
    x="Category",
    y="Sales",
    palette="Set2",
    hue="Category",
    legend=False
)

plt.title("Sales Distribution by Category", fontsize=16)
plt.xlabel("Product Category")
plt.ylabel("Sales Amount")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import accuracy_score, precision_score, f1_score, confusion_matrix, classification_report

full_monthly = monthly_sales.set_index("Order_Month")["Sales"].sort_index().asfreq("MS", fill_value=0).to_frame()
monthly = full_monthly.copy()
monthly["Month"] = monthly.index.month
monthly["Month_Number"] = np.arange(len(monthly))
for lag in [1,2,3,12]:
    monthly[f"Lag_{lag}"] = monthly["Sales"].shift(lag)
monthly["Rolling_Mean_3"] = monthly["Sales"].shift(1).rolling(3).mean()
monthly["Rolling_Mean_6"] = monthly["Sales"].shift(1).rolling(6).mean()
monthly["Sales_Growth"] = monthly["Sales"].pct_change().replace([np.inf,-np.inf],np.nan)
monthly["Next_Sales"] = monthly["Sales"].shift(-1)
monthly = monthly.dropna().copy()
monthly["Target"] = (monthly["Next_Sales"] > monthly["Sales"]).astype(int)
features = ["Month", "Month_Number", "Sales", "Lag_1", "Lag_2", "Lag_3", "Lag_12", "Rolling_Mean_3", "Rolling_Mean_6", "Sales_Growth"]
X, y = monthly[features], monthly["Target"]
split = int(len(monthly)*0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]
print("Training Samples:", len(X_train), "Testing Samples:", len(X_test))
print("Training target counts:", y_train.value_counts().to_dict())

In [ ]:
models = {
 "Logistic Regression": (
    make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=42)),
    {"logisticregression__C":[0.01,0.1,1,10], "logisticregression__class_weight":[None,"balanced"]}),
 "Random Forest": (
    RandomForestClassifier(random_state=42),
    {"n_estimators":[100,200], "max_depth":[2,4,None], "min_samples_leaf":[1,2,3], "class_weight":[None,"balanced"]})
}
# Ensure each training fold contains both classes for Logistic Regression
possible_splits = []
for n_splits in [3,2]:
    cv = TimeSeriesSplit(n_splits=n_splits)
    if all(y_train.iloc[tr].nunique()==2 for tr,_ in cv.split(X_train)):
        possible_splits.append(cv)
cv = possible_splits[0] if possible_splits else None
if y_train.nunique() < 2:
    raise ValueError("Training set contains only one class; cannot train Logistic Regression")
results, predictions_by_model, trained_models = [], {}, {}
for name, (model, params) in models.items():
    print("Training:", name)
    if cv is not None:
        grid = GridSearchCV(model, params, cv=cv, scoring="f1", n_jobs=-1, error_score=np.nan)
        grid.fit(X_train, y_train)
        fitted = grid.best_estimator_
        print("Best Parameters:", grid.best_params_)
    else:
        print("Not enough balanced time folds for tuning; using base model")
        fitted = model.fit(X_train, y_train)
    trained_models[name] = fitted
    prediction = fitted.predict(X_test)
    predictions_by_model[name] = prediction
    results.append({"Model":name,
        "Accuracy":accuracy_score(y_test,prediction),
        "Precision":precision_score(y_test,prediction,zero_division=0),
        "F1_Score":f1_score(y_test,prediction,zero_division=0)})
results_df = pd.DataFrame(results).sort_values(["F1_Score","Accuracy"], ascending=False).reset_index(drop=True)
display(results_df.style.format({"Accuracy":"{:.2%}","Precision":"{:.2%}","F1_Score":"{:.2%}"}))
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]
print("Best Model on held-out test:", best_model_name)
print("Note: Small test size; model rankings are uncertain.")

In [ ]:
best_predictions = predictions_by_model[best_model_name]
print(classification_report(y_test, best_predictions, zero_division=0))
cm = confusion_matrix(y_test, best_predictions, labels=[0,1])
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Not Increased","Increased"], yticklabels=["Not Increased","Increased"])
plt.title("Confusion Matrix - " + best_model_name); plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.tight_layout(); plt.show()
plot_data = results_df.melt(id_vars="Model", value_vars=["Accuracy","Precision","F1_Score"], var_name="Metric", value_name="Score")
plt.figure(figsize=(9,5)); sns.barplot(data=plot_data, x="Metric", y="Score", hue="Model"); plt.ylim(0,1); plt.title("Model Comparison"); plt.tight_layout(); plt.show()
comparison = pd.DataFrame({"Actual":y_test.values, "Predicted":best_predictions}, index=X_test.index)
display(comparison)

In [ ]:
# Retrain selected configuration on all labeled historical months
best_model.fit(X, y)
if len(full_monthly) < 13:
    raise ValueError("Need at least 13 months of history")
prev = full_monthly["Sales"].iloc[-2]
latest_features = pd.DataFrame([{
    "Month":full_monthly.index[-1].month,
    "Month_Number":len(full_monthly)-1,
    "Sales":full_monthly["Sales"].iloc[-1],
    "Lag_1":prev,
    "Lag_2":full_monthly["Sales"].iloc[-3],
    "Lag_3":full_monthly["Sales"].iloc[-4],
    "Lag_12":full_monthly["Sales"].iloc[-13],
    "Rolling_Mean_3":full_monthly["Sales"].iloc[-4:-1].mean(),
    "Rolling_Mean_6":full_monthly["Sales"].iloc[-7:-1].mean(),
    "Sales_Growth":(full_monthly["Sales"].iloc[-1]/prev - 1) if prev != 0 else 0
}])[features]
next_prediction = int(best_model.predict(latest_features)[0])
forecast_month = full_monthly.index[-1] + pd.DateOffset(months=1)
print("Forecast Month:", forecast_month.strftime("%Y-%m"))
print("Predicted Direction:", "INCREASE" if next_prediction else "DECREASE OR SAME")
print("This forecast follows the last month in the historical dataset, not necessarily the current calendar month.")

In [ ]:
# ==========================================
# Evaluate Random Forest Regressor
# ==========================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np
import pandas as pd

# Use the historical training data
X = training_data[features]
y = training_data["Sales"]

# Chronological split: 80% train, 20% test
split = int(len(X) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

# Train the model on historical data only
evaluation_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=5,
    min_samples_leaf=2,
    random_state=42
)

evaluation_model.fit(X_train, y_train)

# Predict test months
y_pred = evaluation_model.predict(X_test)

# Calculate metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Random Forest Regressor Evaluation")
print("---------------------------------")
print(f"MAE: {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R² Score: {r2:.4f}")

# Compare actual and predicted sales
evaluation_results = pd.DataFrame({
    "Actual_Sales": y_test,
    "Predicted_Sales": y_pred
}, index=y_test.index)

display(evaluation_results.round(2))

In [ ]:
# ==========================================
# Linear Regression vs Random Forest Regressor
# ==========================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Prepare historical data
reg_features = [
    "Month",
    "Time_Index",
    "Lag_1",
    "Lag_2",
    "Lag_3",
    "Lag_12"
]

X = training_data[reg_features]
y = training_data["Sales"]

# Chronological split
split = int(len(X) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

# Define models
regression_models = {
    "Linear Regression": LinearRegression(),

    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=300,
        max_depth=5,
        min_samples_leaf=2,
        random_state=42
    )
}

results = []
predictions = {}

# Train and evaluate
for name, model in regression_models.items():

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    predictions[name] = y_pred

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R2": r2_score(y_test, y_pred)
    })

# Comparison table
regression_results = pd.DataFrame(results)

display(
    regression_results.style.format({
        "MAE": "{:,.2f}",
        "RMSE": "{:,.2f}",
        "R2": "{:.4f}"
    })
)

# Best model based on lowest MAE
best_regression = regression_results.loc[
    regression_results["MAE"].idxmin(), "Model"
]

print("Best Regression Model:", best_regression)

# Actual vs predicted chart
plt.figure(figsize=(12, 6))

plt.plot(
    y_test.index,
    y_test.values,
    marker="o",
    label="Actual Sales",
    linewidth=2
)

for name, y_pred in predictions.items():
    plt.plot(
        y_test.index,
        y_pred,
        marker="o",
        linestyle="--",
        label=name
    )

plt.title("Actual vs Predicted Sales - Regression Models")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.legend()
plt.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SALES FORECASTING DASHBOARD - 2027
# Model: Linear Regression
# Historical Data: 2015 - 2018
# ============================================================

# 1. IMPORT LIBRARIES
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from IPython.display import display

%matplotlib inline

sns.set_theme(style="whitegrid")


# ============================================================
# 2. LOAD AND PREPARE DATA
# ============================================================

# Use existing monthly_sales or create it from df

if "monthly_sales" not in globals():

    data = df.copy()

    data["Order_Date"] = pd.to_datetime(
        data["Order_Date"]
    )

    monthly_sales = (
        data.groupby(
            pd.Grouper(
                key="Order_Date",
                freq="MS"
            )
        )["Sales"]
        .sum()
        .reset_index()
        .rename(
            columns={"Order_Date": "Order_Month"}
        )
    )

monthly_sales["Order_Month"] = pd.to_datetime(
    monthly_sales["Order_Month"]
)

history = (
    monthly_sales
    .set_index("Order_Month")["Sales"]
    .sort_index()
    .resample("MS")
    .sum()
    .to_frame()
)

print("Historical Data:")
display(history.head())

print(
    "Last Historical Month:",
    history.index.max().strftime("%Y-%m")
)


# ============================================================
# 3. FEATURE ENGINEERING
# ============================================================

history["Month"] = history.index.month

history["Time_Index"] = np.arange(
    len(history)
)

history["Lag_1"] = history["Sales"].shift(1)
history["Lag_2"] = history["Sales"].shift(2)
history["Lag_3"] = history["Sales"].shift(3)
history["Lag_12"] = history["Sales"].shift(12)

training_data_2027 = history.dropna().copy()

features_2027 = [
    "Month",
    "Time_Index",
    "Lag_1",
    "Lag_2",
    "Lag_3",
    "Lag_12"
]

X = training_data_2027[features_2027]
y = training_data_2027["Sales"]


# ============================================================
# 4. TRAIN AND EVALUATE MODEL
# ============================================================

split = int(len(X) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

evaluation_model = LinearRegression()

evaluation_model.fit(
    X_train,
    y_train
)

y_pred = evaluation_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(
    mean_squared_error(y_test, y_pred)
)

r2 = r2_score(y_test, y_pred)

print("\n================================")
print("LINEAR REGRESSION EVALUATION")
print("================================")

print(f"MAE: {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R2 Score: {r2:.4f}")


# ============================================================
# 5. TRAIN FINAL MODEL
# ============================================================

linear_model_2027 = LinearRegression()

linear_model_2027.fit(X, y)

print("\nLinear Regression Model Trained Successfully!")


# ============================================================
# 6. FORECAST UNTIL DECEMBER 2027
# ============================================================

sales_series = history["Sales"].copy()

future_dates = pd.date_range(
    start=history.index[-1] + pd.offsets.MonthBegin(1),
    end="2027-12-01",
    freq="MS"
)

predictions_2027 = []

for date in future_dates:

    next_features = pd.DataFrame([{
        "Month": date.month,
        "Time_Index": len(sales_series),
        "Lag_1": sales_series.iloc[-1],
        "Lag_2": sales_series.iloc[-2],
        "Lag_3": sales_series.iloc[-3],
        "Lag_12": sales_series.iloc[-12]
    }])

    predicted_sales = linear_model_2027.predict(
        next_features[features_2027]
    )[0]

    predicted_sales = max(
        0,
        predicted_sales
    )

    sales_series.loc[date] = predicted_sales

    predictions_2027.append({
        "Date": date,
        "Predicted_Sales": predicted_sales
    })

forecast_all_2027 = pd.DataFrame(
    predictions_2027
)


# ============================================================
# 7. FILTER YEAR 2027
# ============================================================

forecast_2027_linear = forecast_all_2027[
    forecast_all_2027["Date"].dt.year == 2027
].copy()

forecast_2027_linear["Month"] = (
    forecast_2027_linear["Date"].dt.strftime("%B")
)

print("\n================================")
print("MONTHLY SALES FORECAST - 2027")
print("================================")

display(
    forecast_2027_linear[
        ["Month", "Predicted_Sales"]
    ].style.format({
        "Predicted_Sales": "{:,.2f}"
    })
)


# ============================================================
# 8. FORECAST SUMMARY
# ============================================================

total_2027 = forecast_2027_linear[
    "Predicted_Sales"
].sum()

average_2027 = forecast_2027_linear[
    "Predicted_Sales"
].mean()

highest = forecast_2027_linear.loc[
    forecast_2027_linear["Predicted_Sales"].idxmax()
]

lowest = forecast_2027_linear.loc[
    forecast_2027_linear["Predicted_Sales"].idxmin()
]

print("\n================================")
print("2027 FORECAST SUMMARY")
print("================================")

print(f"Total Predicted Sales: {total_2027:,.2f}")
print(f"Average Monthly Sales: {average_2027:,.2f}")

print(
    f"Highest Month: {highest['Month']} "
    f"({highest['Predicted_Sales']:,.2f})"
)

print(
    f"Lowest Month: {lowest['Month']} "
    f"({lowest['Predicted_Sales']:,.2f})"
)


# ============================================================
# 9. CHART 1 - MONTHLY FORECAST
# ============================================================

plt.figure(figsize=(14, 6))

plt.plot(
    forecast_2027_linear["Date"],
    forecast_2027_linear["Predicted_Sales"],
    marker="o",
    linewidth=3,
    color="#2563EB",
    label="Linear Regression"
)

plt.fill_between(
    forecast_2027_linear["Date"],
    forecast_2027_linear["Predicted_Sales"],
    alpha=0.12,
    color="#2563EB"
)

plt.title(
    "Monthly Sales Forecast - 2027",
    fontsize=17,
    fontweight="bold"
)

plt.xlabel("Month")
plt.ylabel("Predicted Sales")

plt.xticks(
    forecast_2027_linear["Date"],
    forecast_2027_linear["Month"],
    rotation=45
)

plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


# ============================================================
# 10. CHART 2 - MONTHLY SALES BAR CHART
# ============================================================

plt.figure(figsize=(14, 6))

ax = sns.barplot(
    data=forecast_2027_linear,
    x="Month",
    y="Predicted_Sales",
    color="#3B82F6"
)

plt.title(
    "Predicted Sales by Month - 2027",
    fontsize=17,
    fontweight="bold"
)

plt.xlabel("Month")
plt.ylabel("Predicted Sales")

plt.xticks(rotation=45)

for bar in ax.patches:

    height = bar.get_height()

    ax.annotate(
        f"{height:,.0f}",
        (
            bar.get_x() + bar.get_width() / 2,
            height
        ),
        ha="center",
        va="bottom",
        fontsize=8,
        xytext=(0, 4),
        textcoords="offset points"
    )

plt.tight_layout()
plt.show()


# ============================================================
# 11. CHART 3 - HISTORICAL VS FORECAST
# ============================================================

plt.figure(figsize=(15, 7))

plt.plot(
    history.index,
    history["Sales"],
    label="Historical Sales",
    linewidth=2,
    color="#16A34A"
)

plt.plot(
    forecast_all_2027["Date"],
    forecast_all_2027["Predicted_Sales"],
    label="Experimental Forecast 2019-2027",
    linewidth=2,
    linestyle="--",
    color="#F97316"
)

plt.axvline(
    history.index[-1],
    color="gray",
    linestyle=":",
    label="Last Actual Month"
)

plt.title(
    "Historical Sales vs Future Forecast",
    fontsize=17,
    fontweight="bold"
)

plt.xlabel("Year")
plt.ylabel("Sales")

plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


# ============================================================
# 12. CHART 4 - ACTUAL VS PREDICTED (TEST)
# ============================================================

plt.figure(figsize=(13, 6))

plt.plot(
    y_test.index,
    y_test.values,
    marker="o",
    linewidth=2,
    label="Actual Sales"
)

plt.plot(
    y_test.index,
    y_pred,
    marker="o",
    linestyle="--",
    linewidth=2,
    label="Predicted Sales"
)

plt.title(
    "Linear Regression - Actual vs Predicted",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Month")
plt.ylabel("Sales")

plt.xticks(rotation=45)
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


# ============================================================
# 13. EXPORT RESULTS
# ============================================================

forecast_2027_linear[
    ["Date", "Month", "Predicted_Sales"]
].to_csv(
    "Sales_Forecast_2027.csv",
    index=False
)

print("\n================================")
print("FORECAST COMPLETED SUCCESSFULLY!")
print("================================")

print("Saved File: Sales_Forecast_2027.csv")
print("Charts Generated: 4")
print("Forecast Year: 2027")

In [ ]:
import os
print(os.getcwd())

In [ ]:
# Export cleaned sales data
df.to_csv("Sales_Data_Cleaned.csv", index=False)

# Export monthly sales
monthly_sales.to_csv("Monthly_Sales.csv", index=False)

# Export 2027 predictions
forecast_2027_linear.to_csv(
    "Sales_Forecast_2027.csv",
    index=False
)

print("All files exported successfully!")

In [ ]:
import joblib

joblib.dump(model, "trained_model.pkl")